# 01 — EDA del dataset InSDN

Objetivos de este notebook:

1. Cargar el dataset InSDN y comprobar su integridad.
2. Entender distribución de clases (benigno vs cada tipo de ataque).
3. Identificar problemas conocidos: `Infinity`, `NaN`, columnas constantes, posibles fugas de etiqueta.
4. Decidir qué features mantener: enfoque en las **derivables de OpenFlow** (ver `docs/INTERFACE.md`).

**Antes de ejecutar:** descargar el dataset desde [Kaggle](https://www.kaggle.com/datasets/muhammadumarjavaid/insdn-dataset-2020) y descomprimir en `data/raw/`.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Hacer importable src/ desde el notebook
sys.path.insert(0, str(Path.cwd().parent))
from src.features import clean_insdn  # noqa: E402

pd.set_option('display.max_columns', None)
sns.set_theme(style='whitegrid')

## 1. Carga del dataset

InSDN viene típicamente en varios CSV (uno por tipo de tráfico). Vamos a localizarlos automáticamente y concatenarlos.

In [ ]:
DATA_DIR = Path('../data/raw')
csv_files = sorted(DATA_DIR.rglob('*.csv'))
print(f'Encontrados {len(csv_files)} CSV:')
for f in csv_files:
    print(f'  {f.relative_to(DATA_DIR)}  ({f.stat().st_size / 1e6:.1f} MB)')

In [ ]:
frames = [pd.read_csv(f, low_memory=False) for f in csv_files]
df = pd.concat(frames, ignore_index=True)
df.columns = df.columns.str.strip()  # los CSV de CICFlowMeter suelen tener espacios
print(f'Shape: {df.shape}')
df.head()

## 2. Integridad del dataset

In [ ]:
df.dtypes.value_counts()

In [ ]:
n_inf = np.isinf(df.select_dtypes(include=[np.number])).sum().sum()
n_nan = df.isna().sum().sum()
print(f'Valores infinitos: {n_inf}')
print(f'Valores NaN: {n_nan}')
print(f'Filas afectadas (Inf o NaN): {(df.replace([np.inf, -np.inf], np.nan).isna().any(axis=1)).sum()}')

In [ ]:
df_clean = clean_insdn(df)
print(f'Antes: {df.shape}  | Después: {df_clean.shape}')

## 3. Distribución de clases

La columna de etiqueta en InSDN suele llamarse `Label`. Verificad el nombre exacto tras la carga.

In [ ]:
label_candidates = [c for c in df_clean.columns if 'label' in c.lower()]
print('Candidatas a columna de etiqueta:', label_candidates)
LABEL_COL = label_candidates[0] if label_candidates else 'Label'
print(f'Usando: {LABEL_COL}')

In [ ]:
counts = df_clean[LABEL_COL].value_counts()
print(counts)
print(f'\nProporción benigna: {(counts.get("Normal", 0) + counts.get("BENIGN", 0)) / counts.sum():.2%}')

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
counts.plot(kind='bar', ax=ax)
ax.set_yscale('log')
ax.set_ylabel('Número de flows (log)')
ax.set_title('Distribución de clases en InSDN')
plt.tight_layout()

## 4. Posibles fugas de etiqueta

Algunas columnas pueden contener información que no estará disponible en tiempo real desde OpenFlow, o que correlacionan tanto con la etiqueta que delatan el ataque trivialmente (por ejemplo `Destination Port` si los ataques siempre van al mismo puerto víctima).

Comprobamos columnas constantes y columnas con cardinalidad sospechosamente baja.

In [ ]:
nunique = df_clean.nunique().sort_values()
print('Columnas con menos de 5 valores únicos:')
print(nunique[nunique < 5])

## 5. Features compatibles con OpenFlow

Mapear las columnas de InSDN al subconjunto que nuestro detector podrá recibir desde la Ryu app (ver `src/features.py:OPENFLOW_COMPATIBLE_FEATURES`).

Mapping tentativo a confirmar tras ver los nombres reales:

| OpenFlow              | InSDN (típico CICFlowMeter)              |
|-----------------------|------------------------------------------|
| `pkts_per_sec`        | `Flow Pkts/s`                            |
| `bytes_per_sec`       | `Flow Byts/s`                            |
| `avg_pkt_size`        | `Pkt Size Avg`                           |
| `flow_age_sec`        | `Flow Duration` (en µs, convertir)       |
| `src_ip_entropy`      | calcular nosotros sobre ventanas         |
| `dst_port_entropy`    | calcular nosotros sobre ventanas         |
| `new_flows_per_sec`   | calcular nosotros sobre ventanas         |

In [ ]:
for c in df_clean.columns:
    print(c)

## Siguientes pasos

- Cerrar mapping de features OpenFlow ↔ InSDN.
- Guardar versión limpia en `data/processed/insdn_clean.parquet`.
- Pasar a `02_baseline_rf.ipynb` con Random Forest como techo de referencia.